# Notebook 03 — Exploratory Data Analysis (EDA)
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 4 of 8  
**Objective:** Understand patterns, distributions, and relationships in the clean dataset. Identify which features show the strongest association with loan default.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_data
from src.data_cleaner import clean_data
from src.eda import (
    plot_default_rate_by_category,
    plot_numeric_by_default,
    plot_all_numeric_by_default,
    plot_correlation_heatmap,
    plot_default_rate_heatmap,
    plot_feature_boxplots,
    plot_categorical_default_rates,
    plot_binary_default_rates,
    get_default_rates_by_category,
)
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
    FIGURES_DIR, TABLES_DIR
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

---
## 1. Load Clean Data

In [ ]:
df = clean_data(load_data(RAW_DATA_PATH))
print(f'Clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Default rate : {df[TARGET_COLUMN].mean()*100:.2f}%')

---
## 2. Target Variable Distribution

In [ ]:
counts = df[TARGET_COLUMN].value_counts().sort_index()
labels = {0: 'No Default', 1: 'Default'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar([labels[k] for k in counts.index], counts.values,
            color=['#4a90d9', '#e05c5c'], edgecolor='white')
axes[0].set_title('Loan Default Counts', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=[labels[k] for k in counts.index],
            autopct='%1.1f%%', colors=['#4a90d9', '#e05c5c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Default Class Balance', fontweight='bold')

plt.suptitle('Target Variable — Loan Default', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_target_balance.png', bbox_inches='tight')
plt.show()

---
## 3. Numeric Features — Distribution by Default Status

In [ ]:
fig = plot_all_numeric_by_default(df)
fig.savefig(FIGURES_DIR / 'eda_numeric_by_default.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_numeric_by_default.png')

---
## 4. Deep Dive — Key Numeric Features

In [ ]:
for col in ['CreditScore', 'InterestRate', 'DTIRatio', 'Income']:
    fig = plot_numeric_by_default(df, col)
    fig.savefig(FIGURES_DIR / f'eda_deep_{col.lower()}.png', bbox_inches='tight')
    plt.show()

---
## 5. Categorical Features — Default Rate

In [ ]:
fig = plot_categorical_default_rates(df)
fig.savefig(FIGURES_DIR / 'eda_categorical_default_rates.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_categorical_default_rates.png')

---
## 6. Binary Features — Default Rate

In [ ]:
fig = plot_binary_default_rates(df)
fig.savefig(FIGURES_DIR / 'eda_binary_default_rates.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_binary_default_rates.png')

---
## 7. Correlation Heatmap

In [ ]:
fig = plot_correlation_heatmap(df)
fig.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_correlation_heatmap.png')

---
## 8. Correlation with Target — Ranked Table

In [ ]:
corr_target = (
    df[NUMERIC_FEATURES + [TARGET_COLUMN]]
    .corr()[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .abs()
    .sort_values(ascending=False)
    .round(4)
)
print('Absolute correlation with Default (ranked):')
print(corr_target.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
corr_target.plot.barh(ax=ax, color='#4a90d9', edgecolor='white')
ax.set_title('Absolute Correlation with Default — Numeric Features',
             fontweight='bold')
ax.set_xlabel('|Pearson r|')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_with_target.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_correlation_with_target.png')

---
## 9. Cross-Feature Heatmaps — Default Rate

In [ ]:
pairs = [
    ('EmploymentType', 'Education'),
    ('EmploymentType', 'LoanPurpose'),
    ('MaritalStatus',  'LoanPurpose'),
]
for col1, col2 in pairs:
    fig = plot_default_rate_heatmap(df, col1, col2)
    fname = f'eda_heatmap_{col1.lower()}_{col2.lower()}.png'
    fig.savefig(FIGURES_DIR / fname, bbox_inches='tight')
    plt.show()
    print(f'Saved -> outputs/figures/{fname}')

---
## 10. Age Binned Default Rate

In [ ]:
df['AgeBand'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 70],
                       labels=['18-25', '26-35', '36-45', '46-55', '56-69'])
age_rates = get_default_rates_by_category(df, 'AgeBand')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(age_rates['AgeBand'].astype(str), age_rates['default_rate_pct'],
              color='#e05c5c', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Default Rate by Age Band', fontweight='bold')
ax.set_xlabel('Age Band')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, age_rates['default_rate_pct'].max() * 1.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_default_by_age_band.png', bbox_inches='tight')
plt.show()

df.drop(columns=['AgeBand'], inplace=True)
print('Saved -> outputs/figures/eda_default_by_age_band.png')

---
## 11. Income Quartile Default Rate

In [ ]:
df['IncomeQuartile'] = pd.qcut(df['Income'], q=4,
                                labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
inc_rates = get_default_rates_by_category(df, 'IncomeQuartile')
inc_rates = inc_rates.sort_values('IncomeQuartile')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(inc_rates['IncomeQuartile'].astype(str),
              inc_rates['default_rate_pct'],
              color='#7c5cd8', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Default Rate by Income Quartile', fontweight='bold')
ax.set_xlabel('Income Quartile')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, inc_rates['default_rate_pct'].max() * 1.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_default_by_income_quartile.png', bbox_inches='tight')
plt.show()

df.drop(columns=['IncomeQuartile'], inplace=True)
print('Saved -> outputs/figures/eda_default_by_income_quartile.png')

---
## 12. Feature Box Plots (Clean Data)

In [ ]:
fig = plot_feature_boxplots(df)
fig.savefig(FIGURES_DIR / 'eda_feature_boxplots.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/eda_feature_boxplots.png')

---
## 13. Default Rate — All Categorical Features (Summary Table)

In [ ]:
all_rate_tables = []
for col in CATEGORICAL_FEATURES:
    tbl = get_default_rates_by_category(df, col)
    tbl.insert(0, 'Feature', col)
    tbl = tbl.rename(columns={col: 'Value'})
    all_rate_tables.append(tbl)

summary_table = pd.concat(all_rate_tables, ignore_index=True)
summary_table.to_csv(TABLES_DIR / 'default_rates_by_category.csv', index=False)
print('Saved -> outputs/tables/default_rates_by_category.csv')
summary_table

---
## 14. EDA Key Findings

| Finding | Detail |
|---|---|
| Overall default rate | 11.61% — class imbalance present |
| Strongest numeric correlates | *(fill after run — check section 8)* |
| Highest-risk employment type | *(fill after run)* |
| Highest-risk loan purpose | *(fill after run)* |
| Age band with highest default | *(fill after run)* |
| Income quartile with highest default | *(fill after run)* |
| Credit score pattern | *(fill after run)* |
| Interest rate pattern | *(fill after run)* |

---
**Next:** Notebook 04 — KPI Calculations & Risk Analysis